In [5]:
# Step 3: per-gene PC1 of the CpGs (genes with >= 2 CpGs), then restriction to HGNC protein-coding genes.
import io
import time
import urllib.request
from datetime import datetime, timedelta
from multiprocessing import Pool, Value

import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from google.colab import files

MIN_CPGS = 2                 # Methods: genes with >= 2 CpGs
N_GENES_EXPECTED = 359       # Methods: final gene universe
N_SAMPLES_EXPECTED = 2546
HGNC_URLS = [
    "https://storage.googleapis.com/public-download-files/hgnc/tsv/tsv/hgnc_complete_set.txt",
    "https://ftp.ebi.ac.uk/pub/databases/genenames/hgnc/tsv/hgnc_complete_set.txt",
]


def format_time(seconds):
    """Convert seconds to human readable format"""
    return str(timedelta(seconds=int(seconds)))


def validate_cpgs(mapping, array_columns):
    """Keep only genes with at least MIN_CPGS CpGs present in the methylation matrix"""
    valid_mapping = []
    invalid_genes = []

    print("\nValidating CpG IDs...")
    for gene in mapping:
        cpgs = gene[1:]
        valid_cpgs = [cpg for cpg in cpgs if cpg in array_columns]

        if len(valid_cpgs) >= MIN_CPGS:
            valid_mapping.append([gene[0]] + valid_cpgs)
        else:
            invalid_genes.append((gene[0], len(cpgs), len(valid_cpgs)))

    print(f"Original number of genes: {len(mapping)}")
    print(f"Number of valid genes: {len(valid_mapping)}")
    print(f"Number of excluded genes: {len(invalid_genes)}")
    return valid_mapping


def run_PCA(cpg_id):
    # extract CpG signals in current gene, standardise, run PCA
    cpgs = array[cpg_id]
    cpgs = StandardScaler().fit_transform(cpgs)
    transformer = PCA(random_state=1, copy=False, svd_solver='full')
    PCs = transformer.fit_transform(cpgs)
    eigen_vals = transformer.explained_variance_
    return PCs, eigen_vals


def pca_worker(gene):
    with counter.get_lock():
        counter.value += 1
        current = counter.value
    if current % 50 == 0 or current == n_genes:
        print(f"Progress: {current}/{n_genes} genes", flush=True)

    PCs, eigen_vals = run_PCA(gene[1:])
    eigen_vals = np.cumsum(eigen_vals) / np.sum(eigen_vals)
    n_PCs = (eigen_vals > 0.9).sum()
    # gene id, PC1 across samples, variance explained by PC1, number of PCs for 90% variance
    return [gene[0], PCs[:, 0], eigen_vals[0], n_PCs]


# ---------------- Inputs ----------------
print("Upload the methylation matrix WITH probe IDs (methylation_with_probeids.csv, rows = samples, columns = CpGs)...")
uploaded_meth = files.upload()
methylation_file = list(uploaded_meth.keys())[0]
array = pd.read_csv(methylation_file, index_col=0)
print(f"Loaded array with shape: {array.shape}")
assert array.shape[0] == N_SAMPLES_EXPECTED, f"Expected {N_SAMPLES_EXPECTED} samples, got {array.shape[0]}"

print("\nUpload the gene-to-CpG mapping file (gene2DNAm.map)...")
uploaded_map = files.upload()
mapping_file = list(uploaded_map.keys())[0]

mapping = []
with open(mapping_file, 'r') as f:
    for line in f:
        var = line.strip().split()
        if len(var) < MIN_CPGS + 1:      # gene id + at least MIN_CPGS CpGs
            continue
        mapping.append(var)

mapping = validate_cpgs(mapping, array.columns)
if not mapping:
    raise ValueError("No valid genes found after validation. Please check your CpG IDs.")

np.random.seed(10)
n_genes = len(mapping)
n_cpgs = [len(x) - 1 for x in mapping]

log = open('DNAm_PCA_analysis.log', 'w', buffering=1)
log.write(f"Analysis started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")
log.write("Genes with >= %i CpGs: %i\n" % (MIN_CPGS, n_genes))
log.write("No. of CpGs per gene:  Min: %i;  Max: %i;  Median: %i;  Mean: %.2f\n" %
          (np.min(n_cpgs), np.max(n_cpgs), np.median(n_cpgs), np.mean(n_cpgs)))

# ---------------- PC1 per gene ----------------
if __name__ == '__main__':
    print("\nStarting PCA analysis...")
    counter = Value('i', 0)
    start_time = time.time()

    with Pool(10) as pool:
        res = pool.map_async(pca_worker, mapping).get()
    gene_id, PCs, lambda_pc1, lambda_lt_90 = zip(*res)
    print(f"\n\nPCA completed in {format_time(time.time() - start_time)}")

    # samples x genes (PC1 of each gene)
    PCs_df = pd.DataFrame(np.array(PCs).T, index=array.index, columns=list(gene_id))
    PCs_df.to_csv('DNAm_PCA_results.csv', sep=',', index_label='sample_index')
    print(f"All genes with >= {MIN_CPGS} CpGs: {PCs_df.shape} -> DNAm_PCA_results.csv")
    log.write(f"\nPC1 matrix, all genes with >= {MIN_CPGS} CpGs (samples x genes): {PCs_df.shape}\n")

    # ---------------- Restrict to HGNC protein-coding genes ----------------
    hgnc = None
    for u in HGNC_URLS:
        try:
            hgnc = pd.read_csv(io.BytesIO(urllib.request.urlopen(u, timeout=60).read()),
                               sep="\t", low_memory=False)
            print(f"HGNC downloaded from: {u}")
            break
        except Exception as e:
            print(f"Could not download from {u}: {e}")
    if hgnc is None:
        print("Download failed. Download hgnc_complete_set.txt from genenames.org and upload it now.")
        up = files.upload()
        hgnc = pd.read_csv(list(up.keys())[0], sep="\t", low_memory=False)

    protein_coding = set(hgnc.loc[(hgnc["status"] == "Approved") &
                                  (hgnc["locus_group"] == "protein-coding gene"), "symbol"])
    keep = [g for g in PCs_df.columns if g in protein_coding]
    final_df = PCs_df[keep]
    print(f"Protein-coding genes retained: {final_df.shape[1]} of {PCs_df.shape[1]}")

    assert final_df.shape == (N_SAMPLES_EXPECTED, N_GENES_EXPECTED), (
        f"Expected {(N_SAMPLES_EXPECTED, N_GENES_EXPECTED)}, got {final_df.shape}")

    output_file = 'DNAm_imputed_corr_1PC_filt_pcgene_python.csv'
    final_df.to_csv(output_file, sep=',', index_label='sample_index')
    log.write(f"Protein-coding genes retained (HGNC): {final_df.shape[1]}\n")
    log.write(f"Final matrix (samples x genes): {final_df.shape} -> {output_file}\n")
    log.write(f"\nAnalysis completed at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    log.close()

    files.download('DNAm_PCA_results.csv')
    files.download(output_file)
    print(f"Done. Final file: {output_file}  shape: {final_df.shape}")

Upload the methylation matrix WITH probe IDs (methylation_with_probeids.csv, rows = samples, columns = CpGs)...


Saving methylation_with_probeids.csv to methylation_with_probeids.csv
Loaded array with shape: (2546, 2000)

Upload the gene-to-CpG mapping file (gene2DNAm.map)...


Saving gene2DNAm.map to gene2DNAm.map

Validating CpG IDs...
Original number of genes: 447
Number of valid genes: 447
Number of excluded genes: 0

Starting PCA analysis...
Progress: 50/447 genes
Progress: 100/447 genes
Progress: 150/447 genes
Progress: 200/447 genes
Progress: 250/447 genes
Progress: 300/447 genes
Progress: 350/447 genes
Progress: 400/447 genes
Progress: 447/447 genes


PCA completed in 0:00:02
All genes with >= 2 CpGs: (2546, 447) -> DNAm_PCA_results.csv
HGNC downloaded from: https://storage.googleapis.com/public-download-files/hgnc/tsv/tsv/hgnc_complete_set.txt
Protein-coding genes retained: 359 of 447


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Done. Final file: DNAm_imputed_corr_1PC_filt_pcgene_python.csv  shape: (2546, 359)
